# Configuration

MICOM is a full-fledged package for metabolic modeling including a variety of algorithms to build and simulate microbial community models but also to manage metabolic model databases, growth media and taxonomic information. Naturally, there are always small parameters or different approaches that can be used for each step. In newer versions of MICOM those are managed by a global configuration that get attached to each Batch. By default a configuration with sensible defaults is used, but you may change those at any time.

The configuration is divided into categories for each major step in the analysis to make it clear which settings will affect which step. A full documentation of all settings follows here, but we also encourage you to read through the [methods](/logic) section to understand what they mean.

## Configuration settings

### Global

Settings that are used in many places.

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| tolerance           | float | The solver tolerance. Flux values and growth rates are only accurate to this value. |
| solver              | str   | The solver to use. |
| threads             | int   | The number of threads/CPUs to use for calculations. |
| progress            | bool  | Whether to show progress bars. |

### 🔨 Build

Those settings control the building of community models.

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| cutoff              | float | The relative abundance cutoff for taxa. Taxa below that are excluded. |
| force_rebuild       | bool  | Whether to rebuild models if an existing community model is found.    |

### 📊 Simulation

Settings that affect growth simulations.

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| strategy            | str   | The strategy/algorithm to use for growth simulation.             |
| flux_method         | str   | The method to use for estimating likely flux distributions.      |
| host_method         | str   | The method to use for including host models.                     |
| host_growth         | float | The maintenance flux/rate for the host models.                   |
| host_relative       | bool  | Whether to interpret `host_growth` as percentage of the maximum. |

### 🔗 Flux Coupling

Settings that affect flux coupling. This will control how enzyme usage is connected to the growth rate. Enabling this
will ensure that dead (non-growing) organisms can not perform metabolic reactions with high fluxes. 

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| enabled             | bool  | Whether to enable flux coupling.                                 |
| strategy            | str   | The strategy/algorithm to use for flux coupling.                 |
| include_exchanges   | bool  | Whether to include exchange reaction in flux coupling.           |
| constraint          | float | The used coupling constraint.                                    |
| lower               | float | Excess flux allowed (uncoupled flux bound)                       |

### 📁 Databases

Those settings denote the locations of additional databases required by MICOM.

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| microbial           | path/url  | The model database for individual taxa.                      |
| host                | dict      | A dictional of id->path for the host models.                 |
| download_location   | path  | Where to store downloaded databases.                             |

### 🍲 Media

Those settings control the handling of growth media / environmental conditions

| name                | type  | description                                                      |
|---------------------|-------|------------------------------------------------------------------|
| weights             | str   | The weights/scaling used for fluxes when obtaining minimal media. |
| max_import          | float | The maximum added import flux for individual components.         |


## Handling configurations

Configurations are only used by the MICOM Batch API and are thus a part of it. They are handled by the `Configuration` class. When you create a new configuration it will be filled with the default values which should work for most cases.

In [1]:
from micom.batch import Configuration

config = Configuration()
config

# Configuration
  ├ tolerance: 1e-06
  ├ solver: cplex
  ├ threads: 4
  ├ progress: True
  📊 simulation:
     ├ strategy: ctFBA
     ├ tradeoff: 0.5
     ├ flux_method: minimal imports
     ├ host_method: before
     ├ host_relative: True
     ├ host_growth: 0.5
  🔗 coupling:
     ├ enabled: True
     ├ strategy: resource coupling
     ├ include_exchanges: True
     ├ constraint: 400.0
     ├ lower: 0.0
  🔨 build:
     ├ cutoff: 0.0001
     ├ force_rebuild: False
  📁 dbs:
     ├ microbial: default://agora201_refseq216_species_1.qza
     ├ host: None
     ├ download_location: databases
  🍲 media:
     ├ weights: None
     ├ max_import: 10.0

````{tip}
Note the `default://` url prefix in the `dbs` section. Whenever you use this it means that you want to download something from the MICOM default databases. This will work for model databases and media.
````

You can change settings by simply setting them.

In [2]:
config.tolerance = 1e-5
config.coupling.strategy = "enzyme coupling"
config

# Configuration
  ├ tolerance: 1e-05
  ├ solver: cplex
  ├ threads: 4
  ├ progress: True
  📊 simulation:
     ├ strategy: ctFBA
     ├ tradeoff: 0.5
     ├ flux_method: minimal imports
     ├ host_method: before
     ├ host_relative: True
     ├ host_growth: 0.5
  🔗 coupling:
     ├ enabled: True
     ├ strategy: enzyme coupling
     ├ include_exchanges: True
     ├ constraint: 400.0
     ├ lower: 0.0
  🔨 build:
     ├ cutoff: 0.0001
     ├ force_rebuild: False
  📁 dbs:
     ├ microbial: default://agora201_refseq216_species_1.qza
     ├ host: None
     ├ download_location: databases
  🍲 media:
     ├ weights: None
     ├ max_import: 10.0

This will also validate them under the hood.

In [3]:
config.simulation.strategy = "wrong value"

ValidationError: 1 validation error for SimulationConfig
strategy
  Input should be 'ctFBA' or 'SteadyCom' [type=literal_error, input_value='wrong value', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error

Configurations can also be saved and read from YAML or JSON files. In general we recommend to do this for larger studies to maintain a record of the used settings. This is also the recommendation when changing large parts of the configuration: save the default to YAML and update all the parts you want and read this config for you simulations.

In [4]:
config.to_yaml("my_config.yaml")  # can also use .to_json()

In [5]:
config2 = Configuration.from_yaml("my_config.yaml")  # can also use .from_json()
print(config2 == config)

True


## Usage with Batch

In most cases you will never have to handle the `Configuration` object explicitly. A MICOM Batch always comes with a configuration.

In [14]:
from micom import Batch
from micom.data import test_medium, test_db, test_data

tax = test_data()

batch = Batch(
    taxonomy=tax,
    medium=test_medium,
    model_db=test_db,
)

batch.config

# Configuration
  ├ tolerance: 1e-06
  ├ solver: cplex
  ├ threads: 4
  ├ progress: True
  📊 simulation:
     ├ strategy: ctFBA
     ├ tradeoff: 0.5
     ├ flux_method: minimal imports
     ├ host_method: before
     ├ host_relative: True
     ├ host_growth: 0.5
  🔗 coupling:
     ├ enabled: True
     ├ strategy: resource coupling
     ├ include_exchanges: True
     ├ constraint: 400.0
     ├ lower: 0.0
  🔨 build:
     ├ cutoff: 0.0001
     ├ force_rebuild: False
  📁 dbs:
     ├ microbial: /home/cdiener/code/micom/micom/data/artifacts/strain_models.qza
     ├ host: None
     ├ download_location: databases
  🍲 media:
     ├ weights: None
     ├ max_import: 10.0

Note that `Batch` will update the location of the model database if it is supplied. In case you have a saved config you can also provide it directly to Batch.

In [15]:
batch = Batch(
    taxonomy=tax,
    medium=test_medium,
    config=Configuration.from_yaml("my_config.yaml")
)

batch.config

# Configuration
  ├ tolerance: 1e-05
  ├ solver: cplex
  ├ threads: 4
  ├ progress: True
  📊 simulation:
     ├ strategy: ctFBA
     ├ tradeoff: 0.5
     ├ flux_method: minimal imports
     ├ host_method: before
     ├ host_relative: True
     ├ host_growth: 0.5
  🔗 coupling:
     ├ enabled: True
     ├ strategy: enzyme coupling
     ├ include_exchanges: True
     ├ constraint: 400.0
     ├ lower: 0.0
  🔨 build:
     ├ cutoff: 0.0001
     ├ force_rebuild: False
  📁 dbs:
     ├ microbial: default://agora201_refseq216_species_1.qza
     ├ host: None
     ├ download_location: databases
  🍲 media:
     ├ weights: None
     ├ max_import: 10.0

````{warning}
If `model_db` is set it will always overwrite the configuration.
````